In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix
)

from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

import mlflow
import mlflow.sklearn
import mlflow.lightgbm
import mlflow.catboost

In [2]:
PROJECT_ROOT = Path("../")

DATA_PATH = (
    PROJECT_ROOT /
    "data" /
    "features" /
    "customer_churn_dataset.csv"
)


customers_ml = pd.read_csv(DATA_PATH)

customers_ml.head()

,frequency,monetary,avg_order_value,frequency.1,unique_categories,unique_sellers,avg_review_score,late_delivery_ratio,avg_installments,max_installments,payment_method_count,preferred_payment_type,state,latitude,longitude,churn_label
0,2,82.82,41.41,2,2,2,4.5,0.0,1.0,1.0,1,credit_card,SP,-23.577482,-46.587077,1
1,1,141.46,141.46,1,1,1,4.0,0.0,1.0,1.0,1,boleto,BA,-12.186877,-44.540232,0
2,1,179.12,179.12,1,1,1,5.0,0.0,3.0,3.0,1,credit_card,GO,-16.745150,-48.514783,0
3,1,72.20,72.20,1,1,1,5.0,0.0,1.0,1.0,1,credit_card,RN,-5.774002,-35.270976,1
4,1,28.62,28.62,1,1,1,5.0,0.0,1.0,1.0,1,credit_card,SP,-23.676257,-46.514580,0


In [3]:
customers_ml.shape

(96096, 16)

### Train Test Split

In [4]:
X = customers_ml.drop(columns=["churn_label"])

y = customers_ml["churn_label"]

In [5]:
y.value_counts()

churn_label
0    51251
1    44845
Name: count, dtype: int64

In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [7]:
print(f"Training set shape: {X_train.shape}")
print(f"Test set shape: {X_test.shape}")

Training set shape: (76876, 15)
Test set shape: (19220, 15)


### Create Shared Preprocessing Components
- Uses **SimpleImputer** library

In [8]:
print("Categorical features:")
categorical_features = X.select_dtypes(
    include="object"
).columns.tolist()
print(categorical_features)

print("\nNumerical features:")
numeric_features = X.select_dtypes(
    exclude="object"
).columns.tolist()
print(numeric_features)




Categorical features:
['preferred_payment_type', 'state']

Numerical features:
['frequency', 'monetary', 'avg_order_value', 'frequency.1', 'unique_categories', 'unique_sellers', 'avg_review_score', 'late_delivery_ratio', 'avg_installments', 'max_installments', 'payment_method_count', 'latitude', 'longitude']


#### Numerical Imputation

In [9]:
# Missing value imputation for numerical features
numeric_imputer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="median"  # Use median
            )
        )
    ]
)

#### Categorical One-Hot Encoding

In [10]:
# Missing value imputation for categorical features
categorical_encoder = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="most_frequent"   # Use mode
            )
        ),
        (
            "encoder",
            OneHotEncoder(
                handle_unknown="ignore"   # Ignore unknown categories during transformation when deployment
            )
        )
    ]
)

## Model Pipelines

### Logistic Regression Pipeline
- Both numerical and categorical need scaling


- **ColummTransformer** 
  - Applies transformers to columns of an array or pandas DataFrame.
  - Suitable for performing diff processing for diff columns


                  Dataset
                     |
          ┌──────────┴──────────┐
          ↓                     ↓
     Numerical              Categorical
          ↓                     ↓
     Imputation             Imputation
          ↓                     ↓
      Scaling              One-Hot Encoding
          └──────────┬──────────┘
                     ↓
              Model input

#### Expected flow will be like:
                 DATA
                   ↓
             Train/Test Split
                   ↓
             Preprocessing
                   ↓
        ┌──────────┼──────────┐
        ↓          ↓          ↓
      Model      Model      Model
        ↓          ↓          ↓
        └──────────┼──────────┘
                   ↓
                Evaluate
                   ↓
               MLflow Run
                   ↓
          Compare Experiments
                   ↓
            Select Best Model
                   ↓
             Model Registry
                   ↓
                Deploy

In [11]:
logistic_preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            Pipeline(
                steps=[
                    ("imputer", SimpleImputer(strategy="median")),
                    ("scaler", StandardScaler())  # Standardize numeric features into Standard Normal Distribution
                ]
            ),
            numeric_features
        ),

        (
            "cat",
            categorical_encoder,
            categorical_features
        )
    ]
)

### Tree Model Pipeline
- LightGBM and Random Forest

In [12]:
tree_preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            numeric_imputer,
            numeric_features
        ),

        (
            "cat",
            categorical_encoder,
            categorical_features
        )
    ]
)

### Evaluataion Function

In [13]:
def evaluate_model(model, X_test, y_test):

    predictions = model.predict(X_test)  # Returns the final class

    probabilities = model.predict_proba(X_test)[:, 1]  # Returns probabilities for the 
                                                       # positive class (churn)

    metrics = {
        "accuracy": accuracy_score(
            y_test,
            predictions
        ),

        "precision": precision_score(
            y_test,
            predictions
        ),

        "recall": recall_score(
            y_test,
            predictions
        ),

        "f1_score": f1_score(
            y_test,
            predictions
        ),

        "roc_auc": roc_auc_score(
            y_test,
            probabilities
        )
    }

    return metrics

### Setup ML workflow

In [14]:
from pathlib import Path
import mlflow

PROJECT_ROOT = Path("../").resolve()

mlflow.set_tracking_uri(
    f"sqlite:///{PROJECT_ROOT / 'mlflow.db'}"
)

mlflow.set_experiment(
    "ecom_customer_churn_prediction"
)

<Experiment: artifact_location='file:///c:/Users/cjc72/Desktop/e-commerce-ai-ml-retention-platform/e-commerce-ai-ml-retention-platform/notebooks/mlruns/1', creation_time=1786162743100, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1786162743100, lifecycle_stage='active', name='ecom_customer_churn_prediction', tags={}, trace_location=None, workspace='default'>

In [15]:
import mlflow

print(mlflow.get_tracking_uri())

sqlite:///C:\Users\cjc72\Desktop\e-commerce-ai-ml-retention-platform\e-commerce-ai-ml-retention-platform\mlflow.db


## Model Training

- **MLflow** describes a run as an execution of data-science code, with metadata such as parameters, metrics, timing information and artifacts.

- **Artifacts** = files/output associated with a run, like trained models, images and other output files.

### Logistic Regression

In [ ]:
params = {
    "max_iter": 1000,
    "class_weight": "balanced"  # Handle imbalance classes
}

# Creates a run in the experiment
with mlflow.start_run(     
    run_name="logistic_regression"   
):

    model = Pipeline(
        steps=[
            (
                "preprocessor",
                logistic_preprocessor
            ),
            (
                "model",
                LogisticRegression(
                    **params
                )
            )
        ]
    )


    model.fit(
        X_train,
        y_train
    )


    metrics = evaluate_model(
        model,
        X_test,
        y_test
    )

    print(**params)
    print(metrics)

    mlflow.log_params(params)     # Logs the parameters to the MLflow run
    mlflow.log_metrics(metrics)   # Logs the metrics to the MLflow run


    mlflow.sklearn.log_model(    # Save model as an MLflow model artifact.
        model,
        "model",
        serialization_format="pickle"
    )

2026/08/09 01:03:47 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/08/09 01:03:47 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


{'max_iter': 1000, 'class_weight': 'balanced'}
{'accuracy': 0.5429240374609782, 'precision': 0.509064039408867, 'recall': 0.5760954398483666, 'f1_score': 0.5405094408703384, 'roc_auc': 0.567307237899467}


### Random Forest

In [21]:
params = {
    "n_estimators": 300,
    "class_weight": "balanced"
}

# Creates a run in the experiment
with mlflow.start_run(
    run_name="random_forest"
):

    model = Pipeline(
        steps=[
            (
                "preprocessor",
                tree_preprocessor
            ),
            (
                "model",
                RandomForestClassifier(
                    **params
                )
            )
        ]
    )


    model.fit(
        X_train,
        y_train
    )


    metrics = evaluate_model(
        model,
        X_test,
        y_test
    )

    print(params)
    print(metrics)

    mlflow.log_params(params)     # Logs the parameters to the MLflow run
    mlflow.log_metrics(metrics)   # Logs the metrics to the MLflow run

    mlflow.sklearn.log_model(    # Save model as an MLflow model artifact.
        model,
        "model",
        serialization_format="pickle"
    )

2026/08/09 01:05:58 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/08/09 01:05:58 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


{'n_estimators': 300, 'class_weight': 'balanced'}
{'accuracy': 0.5563995837669095, 'precision': 0.5249915378539998, 'recall': 0.5187869327684246, 'f1_score': 0.5218707940780619, 'roc_auc': 0.5805115276968429}


### LightGBM

In [22]:
params = {
    "n_estimators": 300,
    "learning_rate": 0.05,
    "random_state": 42,
    "class_weight": "balanced"
}

# Creates a run in the experiment
with mlflow.start_run(
    run_name="lightgbm"
):

    model = Pipeline(
        steps=[
            (
                "preprocessor",
                tree_preprocessor
            ),
            (
                "model",
                LGBMClassifier(
                    **params
                )
            )
        ]
    )


    model.fit(
        X_train,
        y_train
    )


    metrics = evaluate_model(
        model,
        X_test,
        y_test
    )


    print(params)
    print(metrics)

    mlflow.log_params(params)     # Logs the parameters to the MLflow run
    mlflow.log_metrics(metrics)   # Logs the metrics to the MLflow run


    mlflow.sklearn.log_model(    # Save model as an MLflow model artifact.
        model,
        "model",
        serialization_format="pickle"
    )

[LightGBM] [Info] Number of positive: 35876, number of negative: 41000
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.004819 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1257
[LightGBM] [Info] Number of data points in the train set: 76876, number of used features: 45
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Start training from score 0.000000


2026/08/09 01:06:14 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/08/09 01:06:14 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


{'n_estimators': 300, 'learning_rate': 0.05, 'random_state': 42, 'class_weight': 'balanced'}
{'accuracy': 0.5841311134235172, 'precision': 0.5519038502446288, 'recall': 0.5785483331475081, 'f1_score': 0.5649120897066028, 'roc_auc': 0.6167555109313919}


### CatBoost
- It can handle categorical variables so no need encoding

In [23]:
catboost_X_train = X_train.copy()

catboost_X_test = X_test.copy()


cat_features_index = [
    catboost_X_train.columns.get_loc(col)

    for col in categorical_features
]

In [24]:
params = {
    "iterations": 300,
    "learning_rate": 0.05,
    "depth": 6,
    "random_seed": 42,
    "verbose": 0,
    "auto_class_weights": "Balanced"
}

# Creates a run in the experiment
with mlflow.start_run(
    run_name="catboost"
):

    model = CatBoostClassifier(
        **params
    )


    model.fit(
        catboost_X_train,
        y_train,
        cat_features=cat_features_index
    )


    metrics = evaluate_model(
        model,
        catboost_X_test,
        y_test
    )


    print(params)
    print(metrics)

    mlflow.log_params(params)     # Logs the parameters to the MLflow run
    mlflow.log_metrics(metrics)   # Logs the metrics to the MLflow run


    mlflow.sklearn.log_model(    # Save model as an MLflow model artifact.
        model,
        "model",
        serialization_format="pickle"
    )

2026/08/09 01:06:47 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/08/09 01:06:47 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


{'iterations': 300, 'learning_rate': 0.05, 'depth': 6, 'random_seed': 42, 'verbose': 0, 'auto_class_weights': 'Balanced'}
{'accuracy': 0.5681581685744017, 'precision': 0.534876446668752, 'recall': 0.5719701192998105, 'f1_score': 0.552801724137931, 'roc_auc': 0.5982927254858347}


After running the cells, 
1. Run in terminal:
- mlflow ui
2. Open:
- http://localhost:5000